<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/01-python-foundations/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: A k-Nearest Neighbours Classifier from Scratch

In this project you'll build a working machine learning classifier using only what's in Module 1: lists, loops, functions, comprehensions, dictionaries and `sorted` with a `key`. No libraries.

The goal is to classify fruit as an apple, an orange or a lemon from two measurements: its weight and how sweet it is. Along the way you'll run into one of the most common mistakes in real ML projects (forgetting to scale your features) and see exactly why it matters.

By the end you'll have:

1. a synthetic dataset
2. a shuffled train/test split
3. a distance function
4. a k-NN classifier
5. an evaluation loop that compares different values of k
6. a feature scaler, and a before-and-after comparison

## How k-NN works

k-nearest neighbours is probably the simplest classifier there is, which makes it a good first one to build.

To classify a new fruit:

1. Measure its distance to every fruit in the training set.
2. Take the `k` closest ones (its "neighbours").
3. Let them vote. The most common label among the neighbours wins.

There's no training step at all. The "model" is just the training data. That makes k-NN easy to understand, but slow to predict with on big datasets, since every prediction compares against every training example. We'll come back to that at the end.

## Step 1: Create the dataset

We'll generate a synthetic dataset so we know exactly what's in it. Each fruit has two features:

- `weight` in grams
- `sweetness`, a score from 0 to 15

The numbers are made up but roughly realistic. Apples and oranges weigh about the same, so weight alone can't tell them apart, but they differ in sweetness. Lemons are light and sour, so they're easy to spot.

Each row is a tuple: `(features, label)`, where `features` is a list `[weight, sweetness]`.

In [1]:
import random

random.seed(7)


def make_fruit(label, weight_mean, weight_sd, sweet_mean, sweet_sd):
    weight = random.gauss(weight_mean, weight_sd)
    sweetness = random.gauss(sweet_mean, sweet_sd)
    return ([round(weight, 1), round(sweetness, 2)], label)


dataset = []
for _ in range(60):
    dataset.append(make_fruit("apple", 170, 20, 13.0, 0.8))
    dataset.append(make_fruit("orange", 175, 25, 10.5, 0.8))
    dataset.append(make_fruit("lemon", 90, 15, 3.0, 0.8))

print(len(dataset), "fruits")
for row in dataset[:6]:
    print(row)

180 fruits
([164.9, 13.41], 'apple')
([169.3, 10.25], 'orange')
([76.0, 2.83], 'lemon')
([192.2, 13.34], 'apple')
([200.9, 10.7], 'orange')
([95.9, 3.15], 'lemon')


Let's check the average weight and sweetness per class, to make sure the data looks the way we intended:

In [2]:
def column_mean(rows, j):
    return sum(features[j] for features, _ in rows) / len(rows)


for name in ["apple", "orange", "lemon"]:
    rows = [row for row in dataset if row[1] == name]
    print(f"{name:7s} n={len(rows)}  weight={column_mean(rows, 0):6.1f} g  "
          f"sweetness={column_mean(rows, 1):5.2f}")

apple   n=60  weight= 169.0 g  sweetness=13.00
orange  n=60  weight= 176.2 g  sweetness=10.59
lemon   n=60  weight=  92.2 g  sweetness= 2.98


## Step 2: Shuffle and split

Look at how the dataset was built: apple, orange, lemon, apple, orange, lemon... Real data often comes in some order too (sorted by date, by class, by whoever collected it). If we split without shuffling, the test set might not look like the training set at all.

We shuffle a copy so the original list stays untouched. `random.Random(seed)` creates a separate random generator with its own seed, so the split is the same every time you run the notebook, no matter what other random calls happened before.

In [3]:
def train_test_split(data, test_ratio, seed=0):
    rows = data.copy()
    random.Random(seed).shuffle(rows)
    n_test = round(len(rows) * test_ratio)
    return rows[n_test:], rows[:n_test]


train, test = train_test_split(dataset, test_ratio=0.25)
print(f"train: {len(train)} fruits, test: {len(test)} fruits")

train: 135 fruits, test: 45 fruits


It's worth checking that every class ended up in both sets in reasonable numbers:

In [4]:
def count_labels(rows):
    counts = {}
    for _, label in rows:
        counts[label] = counts.get(label, 0) + 1
    return counts


print("train:", count_labels(train))
print("test: ", count_labels(test))

train: {'lemon': 41, 'apple': 51, 'orange': 43}
test:  {'lemon': 19, 'orange': 17, 'apple': 9}


## Step 3: Measure distance

To find the "nearest" neighbours we need a way to measure how far apart two fruits are. The usual choice is Euclidean distance, the straight-line distance you know from geometry:

$$d(a, b) = \sqrt{\sum_j (a_j - b_j)^2}$$

In [5]:
def euclidean(a, b):
    return sum((x - y) ** 2 for x, y in zip(a, b)) ** 0.5


print(euclidean([0, 0], [3, 4]))   # should be 5.0

5.0


## Step 4: Vote

Given the labels of the neighbours, return the most common one. This is the same counting idea as `count_labels`, followed by `max` with a `key`.

In [6]:
def majority_vote(labels):
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return max(counts, key=counts.get)


print(majority_vote(["apple", "orange", "apple"]))
print(majority_vote(["lemon", "apple", "orange", "apple", "orange"]))

apple
apple


Look at the second example. Apple and orange both have two votes, and apple wins. Why?

`max` returns the first item with the highest value, and dictionaries remember insertion order, so the tie goes to whichever label appeared first in the list. Since we'll pass in neighbours sorted from closest to farthest, a tie goes to the class whose nearest member is closest. That's a sensible rule, and we got it for free. It's still a good idea to use an odd `k` to make ties less likely.

## Step 5: Predict

Now put the pieces together. Sort the training set by distance to the new point, take the first `k`, and vote.

In [7]:
def predict(train_rows, features, k=3):
    by_distance = sorted(train_rows, key=lambda row: euclidean(row[0], features))
    nearest_labels = [label for _, label in by_distance[:k]]
    return majority_vote(nearest_labels)


print(predict(train, [95.0, 3.2]))     # light and sour
print(predict(train, [165.0, 13.4]))   # heavy and sweet

lemon
apple


## Step 6: Evaluate

Accuracy on the test set tells us how well the classifier does on fruit it has never seen. Let's try a few values of `k`.

In [8]:
def evaluate(train_rows, test_rows, k):
    correct = sum(predict(train_rows, features, k) == label
                  for features, label in test_rows)
    return correct / len(test_rows)


for k in [1, 3, 5, 9, 15]:
    print(f"k={k:2d}  accuracy={evaluate(train, test, k):.1%}")

k= 1  accuracy=88.9%
k= 3  accuracy=77.8%
k= 5  accuracy=71.1%
k= 9  accuracy=68.9%
k=15  accuracy=73.3%


Not bad, but not great either. Lemons are easy, so something else is going wrong. Let's look at which fruits it gets wrong:

In [9]:
k = 5
mistakes = [(features, label, predict(train, features, k))
            for features, label in test
            if predict(train, features, k) != label]

print(f"{len(mistakes)} mistakes out of {len(test)}")
for features, label, guess in mistakes:
    print(f"  {features}  true={label:7s} predicted={guess}")

13 mistakes out of 45
  [135.8, 11.53]  true=orange  predicted=apple
  [179.4, 11.46]  true=orange  predicted=apple
  [183.7, 11.81]  true=apple   predicted=orange
  [176.2, 11.54]  true=orange  predicted=apple
  [149.6, 10.21]  true=orange  predicted=apple
  [196.4, 11.69]  true=orange  predicted=apple
  [197.9, 10.29]  true=orange  predicted=apple
  [145.1, 9.42]  true=orange  predicted=apple
  [153.2, 10.71]  true=orange  predicted=apple
  [152.4, 10.43]  true=orange  predicted=apple
  [138.6, 10.69]  true=orange  predicted=apple
  [136.8, 9.29]  true=orange  predicted=apple
  [125.7, 11.63]  true=orange  predicted=apple


Every mistake is an apple and an orange being confused with each other, even though their sweetness is clearly different. To see why, compare these distances:

In [10]:
apple = [170.0, 13.0]
orange = [178.0, 10.5]       # different fruit, similar weight
heavy_apple = [200.0, 13.0]  # same kind of fruit, just bigger

print(f"apple to orange:      {euclidean(apple, orange):.1f}")
print(f"apple to heavy apple: {euclidean(apple, heavy_apple):.1f}")

apple to orange:      8.4
apple to heavy apple: 30.0


According to our distance function, the orange is much "closer" to the apple than the other apple is.

The problem is the scale of the features. Weight varies by tens of grams, while sweetness only varies by a few points. When you add up squared differences, the weight term swamps the sweetness term, so the classifier is basically ignoring sweetness, which is the one feature that separates apples from oranges.

This isn't a k-NN quirk. Any algorithm that uses distances (k-NN, k-means, SVMs) or gradient descent (linear models, neural networks) is affected by feature scale.

## Step 7: Scale the features

The fix is to put every feature on the same scale. We'll use min-max scaling, which you wrote in the exercises: map each feature so its minimum becomes 0 and its maximum becomes 1.

**The important part:** compute the min and max from the **training set only**, then apply those same numbers to the test set. In real life you don't have the test data when you build your model. If your preprocessing looks at it, you're leaking information about the test set into training, and your accuracy estimate will be too optimistic. This is called data leakage, and it's one of the most common mistakes in ML projects.

So the scaler has two parts: `fit_minmax` learns the numbers from the training data, and `transform` applies them to any data.

In [11]:
def fit_minmax(rows):
    n_features = len(rows[0][0])
    mins = [min(features[j] for features, _ in rows) for j in range(n_features)]
    maxs = [max(features[j] for features, _ in rows) for j in range(n_features)]
    return mins, maxs


def scale(features, mins, maxs):
    return [(x - lo) / (hi - lo) for x, lo, hi in zip(features, mins, maxs)]


def transform(rows, mins, maxs):
    return [(scale(features, mins, maxs), label) for features, label in rows]


mins, maxs = fit_minmax(train)
print("mins:", mins)
print("maxs:", maxs)

train_scaled = transform(train, mins, maxs)
test_scaled = transform(test, mins, maxs)

print(train[0])
print(train_scaled[0])

mins: [59.8, 0.93]
maxs: [241.7, 14.94]
([104.3, 3.42], 'lemon')
([0.2446399120395822, 0.17773019271948606], 'lemon')


Some test values might come out slightly below 0 or above 1, because the test set can contain a fruit that's a bit lighter or heavier than anything in training. That's fine and expected.

Now the same comparison as before, with and without scaling:

In [12]:
print(" k   unscaled   scaled")
for k in [1, 3, 5, 9, 15]:
    before = evaluate(train, test, k)
    after = evaluate(train_scaled, test_scaled, k)
    print(f"{k:2d}   {before:7.1%}   {after:7.1%}")

 k   unscaled   scaled
 1     88.9%     93.3%
 3     77.8%     95.6%
 5     71.1%     93.3%
 9     68.9%     93.3%
15     73.3%     95.6%


Same data, same algorithm, same `k`. The only change is putting the features on a comparable scale, and the accuracy jumps.

This is why every serious ML pipeline has a preprocessing step, and why scikit-learn's scalers have separate `fit` and `transform` methods, exactly like the two functions we just wrote.

## Step 8: Classify a new fruit

To classify a new measurement, scale it with the training min and max first, then predict:

In [13]:
def classify(weight, sweetness, k=5):
    features = scale([weight, sweetness], mins, maxs)
    return predict(train_scaled, features, k)


for weight, sweetness in [(172, 13.1), (180, 10.2), (88, 2.5), (150, 11.8)]:
    print(f"{weight} g, sweetness {sweetness}: {classify(weight, sweetness)}")

172 g, sweetness 13.1: apple
180 g, sweetness 10.2: orange
88 g, sweetness 2.5: lemon
150 g, sweetness 11.8: apple


## Your turn

The classifier works, but there's plenty of room to experiment. Try some of these:

1. **Manhattan distance.** Write `manhattan(a, b)`, which sums the absolute differences instead of the squared ones. Swap it into `predict` and compare the accuracy. Better yet, give `predict` a `distance` argument so you can pass in either function, the same way we passed metrics around in the lesson.

2. **Weighted voting.** Right now the closest neighbour and the k-th closest get the same vote. Give each neighbour a vote of `1 / distance` instead. Watch out for a distance of exactly 0.

3. **Standardization.** Instead of min-max scaling, transform each feature to have mean 0 and standard deviation 1, using your `mean_and_std` function from the exercises. Does it make a difference here?

4. **Pick k properly.** We chose `k` by looking at test accuracy, which is actually cheating: we used the test set to make a decision about the model. Split the training set again into a smaller training set and a validation set, choose `k` on the validation set, and only touch the test set once at the very end.

5. **Feel the slowness.** Change the dataset loop from 60 to 2000 fruits per class and time `evaluate` with the `time` module. Every prediction sorts the whole training set. In Module 6 we'll rewrite the distance computation with NumPy and see how much faster it gets.

## What you used

Looking back, this project used almost everything from Module 1:

- **lists and tuples** to store the data
- **functions** with default and keyword arguments (`k=5`, `seed=0`)
- **comprehensions** for scaling, filtering and collecting labels
- **dictionaries** for counting votes
- **`sorted` and `max` with a `key`** for finding neighbours and the winning label
- **`zip`** everywhere features are compared or scaled
- **`.copy()`** to shuffle without touching the original data

And some ideas you'll keep meeting in ML: train/test splits, shuffling, feature scaling, fit vs transform, data leakage and hyperparameter choice.